In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("ambulance_updated_dataset (3).csv")

# FIX datetime
df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['datetime'])
df = df.sort_values(by='datetime')

df.head()

,datetime,latitude,longitude,emergency_type,hour,day,traffic_level,temperature,rain,demand
0,2015-12-10 17:40:00,40.297876,-75.581294,EMS,17,Thursday,0.562178,38,0,1
15,2015-12-10 17:40:00,40.223778,-75.235399,Traffic,17,Thursday,0.428383,31,1,1
14,2015-12-10 17:40:00,40.097222,-75.376195,Traffic,17,Thursday,0.427277,23,0,1
13,2015-12-10 17:40:00,40.062974,-75.135914,Traffic,17,Thursday,0.448637,23,0,1
12,2015-12-10 17:40:00,40.174131,-75.098491,Traffic,17,Thursday,0.882710,30,1,1


In [6]:
# Past Demand
df['past_1hr_demand'] = df['demand'].rolling(window=60).sum()
df['past_1hr_demand'].fillna(0, inplace=True)

# Peak Hour
df['peak_hour'] = df['hour'].apply(
    lambda x: 1 if (8 <= x <= 11 or 17 <= x <= 21) else 0
)

/tmp/ipykernel_10131/2472204059.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['past_1hr_demand'].fillna(0, inplace=True)


In [7]:
X = df.drop(columns=['datetime', 'demand'])
y = df['demand']

X.head()

,latitude,longitude,emergency_type,hour,day,traffic_level,temperature,rain,past_1hr_demand,peak_hour
0,40.297876,-75.581294,EMS,17,Thursday,0.562178,38,0,NaN,1
15,40.223778,-75.235399,Traffic,17,Thursday,0.428383,31,1,NaN,1
14,40.097222,-75.376195,Traffic,17,Thursday,0.427277,23,0,NaN,1
13,40.062974,-75.135914,Traffic,17,Thursday,0.448637,23,0,NaN,1
12,40.174131,-75.098491,Traffic,17,Thursday,0.882710,30,1,NaN,1


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# categorical + numeric columns
cat_cols = ['emergency_type', 'day']
num_cols = [col for col in X.columns if col not in cat_cols]

# transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

# full pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100))
])

In [9]:
from sklearn.model_selection import train_test_split

xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(xtrain, ytrain)

print("Train Accuracy:", model.score(xtrain, ytrain))
print("Test Accuracy:", model.score(xtest, ytest))

Train Accuracy: 0.9999874360810624
Test Accuracy: 0.9505000251268908


In [10]:
import pickle

with open("final_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [11]:
import pickle
import pandas as pd

def smart_prediction(input_dict):

    model = pickle.load(open("final_model.pkl", "rb"))

    # convert to dataframe
    data = pd.DataFrame([input_dict])

    prediction = model.predict(data)[0]

    traffic = input_dict['traffic_level']

    if prediction == 1 and traffic > 0.7:
        ambulances = 3
        level = "HIGH"
    elif prediction == 1:
        ambulances = 2
        level = "MEDIUM"
    else:
        ambulances = 1
        level = "LOW"

    response_time = 10 + (traffic * 10)

    return {
        "Demand": int(prediction),
        "Priority": level,
        "Ambulances": ambulances,
        "Response Time": round(response_time, 2)
    }

In [12]:
sample_input = {
    "latitude": 40.2,
    "longitude": -75.3,
    "emergency_type": "EMS",
    "hour": 18,
    "day": "Thursday",
    "traffic_level": 0.8,
    "temperature": 32,
    "rain": 5,
    "past_1hr_demand": 10,
    "peak_hour": 1
}

print(smart_prediction(sample_input))

{'Demand': 1, 'Priority': 'HIGH', 'Ambulances': 3, 'Response Time': 18.0}
